# Zein helix 84-115 - kappa-casein fragment (64 aa, compact rank_004) Protein-Protein Docking (MEGADOCK, GPU-accelerated)

Companion to the full-length run in `../full_zein_length/`. **Same software, same parameters, same
analysis** - only the zein receptor changed, so the two sets of results are directly comparable.

## What changed, and why

The full-length AlphaFold model of alpha-zein is not reproducible. Pairwise C-alpha RMSD between the five
ColabFold models is **13.7-21.5 A** - they are effectively five different molecules, so any docking
result computed on one of them is specific to an arbitrary conformer.

Restricted to **residues 84-115**, the same comparison gives **0.3-1.1 A**. All five predictions
converge on this helix. It is the one part of alpha-zein that AlphaFold genuinely resolves, and therefore
the only defensible receptor.

| | Full-length model | Helix 84-115 |
|---|---|---|
| Residues | 234 | 32 |
| Mean pLDDT | 49.0 | 55-78 |
| Cross-model RMSD | 13.7-21.5 A | **0.3-1.1 A** |
| Restraint residues | 40, spanning **92.8 A** (scattered) | 15, one **contiguous** solvent-facing stripe |

**Sequence:** `LPLVHLLAQNIRAQQLQQLVLANLAAYSQQQQ` - a single alpha-helix, the canonical Leu/Gln-rich
alpha-zein repeat.

## Why the restraint list changed too

In the full-length run the 40 allowed residues were scattered over 92.8 A and did not form a contiguous
patch. After blocking the other 194, the receptor presented a "spotty wall" and ligands could only perch
on 3-6 residues - roughly an order of magnitude smaller than a normal protein-protein interface, which is
why all five ligands scored alike.

Here the kept residues are the **solvent-facing stripe of one helical face** (15 contiguous-in-space
residues). On a real nanoparticle the opposite face packs into the particle interior, so blocking it
encodes the nanoparticle geometry properly instead of arbitrarily.

> **Input changed from the full-length run.** The original `kappa_casein_frag.pdb` (AlphaFold rank_001) was a near-straight rod (Rg 49.0 A, end-to-end 164.8 A) that forced MEGADOCK onto a 294 grid instead of 192, making its score incomparable with the other four. This notebook uses **rank_004** (Rg 27.3 A, end-to-end 31.1 A), which is close to random-coil dimensions and fits the standard grid.

**License note:** MEGADOCK is CC BY-NC 4.0 - free for academic/research use, commercial use requires
permission from Tokyo Institute of Technology.


## 1. Set up Colab runtime

Before running anything: **Runtime -> Change runtime type -> GPU** (T4 is fine).

In [ ]:
# @title Install MEGADOCK and dependencies (GPU build)

# Clone MEGADOCK
!git clone https://github.com/akiyamalab/MEGADOCK

# FFT library required by MEGADOCK
!apt-get install -y libfftw3-dev libfftw3-single3

# Build the GPU binary
%cd /content/MEGADOCK
!make -j 2 -f Makefile.colab

# Python deps for structure handling + visualization
!pip install -q biopython
!pip install -q nglview==3.0.8
!jupyter-nbextension enable nglview --py --sys-prefix


## 2. Connect Google Drive and locate your structures

**One-time setup:**

1. Go to [Google Drive](https://drive.google.com).
2. Create a folder named `zein84_115_casein`.
3. Upload these 3 files into it, flat, no subfolders:
   - `zein_helix_84_115.pdb`   (local: `protein_structures/zein_84-115/`)
   - `zein_helix_restricted.txt`   (local: `protein_structures/zein_84-115/`)
   - `kappa_casein_frag_compact_rank004.pdb`   (local: `protein_structures/casein/`)

In [ ]:
# @title Mount Google Drive and copy the structure files into the working directory
from google.colab import drive
drive.mount('/content/drive')

# Change this if you used a different folder name / location in your Drive
DRIVE_FOLDER = "/content/drive/MyDrive/zein84_115_casein"

import os, shutil

required_files = ["zein_helix_84_115.pdb", "zein_helix_restricted.txt", "kappa_casein_frag_compact_rank004.pdb"]
missing = [f for f in required_files if not os.path.exists(os.path.join(DRIVE_FOLDER, f))]

if missing:
    raise FileNotFoundError(
        f"Missing {missing} in {DRIVE_FOLDER}. Upload them to that Drive folder (see instructions above), "
        f"or update DRIVE_FOLDER to point at the correct path, then re-run this cell."
    )

for fname in required_files:
    shutil.copy(os.path.join(DRIVE_FOLDER, fname), f"/content/MEGADOCK/{fname}")

print("Copied from Drive into /content/MEGADOCK:")
for fname in required_files:
    print(" -", fname)


## 3. Build the restraint-blocked zein helix receptor

Identical mechanism to the full-length notebook: MEGADOCK's `block` marks excluded residues as `BLK` so
the FFT search ignores them. Only the kept list differs.

**Kept (solvent-facing stripe, 15 residues):** `84,85,88,91,92,95,96,99,102,103,106,107,110,113,114`

**Blocked (17):** the opposite helical face, which on a real nanoparticle packs into the particle interior.

Set `APPLY_BLOCK = False` to run the unrestricted control - useful for checking how much the face
assignment is driving the result.

In [ ]:
# @title Compute the blocked-residue list and apply blocking (pure Python 3)
#
# MEGADOCK's `block` tool ships as a Python 2 script and won't run on modern Colab
# images, so its logic (rename the resName field to "BLK") is reimplemented here.

APPLY_BLOCK = True   # set False for the unrestricted control run

RECEPTOR_PDB = "zein_helix_84_115.pdb"
RESTRAINTS_FILE = "zein_helix_restricted.txt"
BLOCKED_RECEPTOR = "zein_helix_blocked.pdb" if APPLY_BLOCK else "zein_helix_84_115.pdb"

with open(RESTRAINTS_FILE) as f:
    line = f.read().strip()
keep_str, chain = line.split(":")
keep_residues = set(int(x) for x in keep_str.split(","))

all_residues = set()
with open(RECEPTOR_PDB) as f:
    for l in f:
        if l.startswith("ATOM") and l[21].strip() == chain:
            all_residues.add(int(l[22:26]))

block_residues = all_residues - keep_residues

print(f"Chain: {chain}")
print(f"Total residues: {len(all_residues)}")
print(f"Keeping (solvent-facing stripe): {len(keep_residues)} -> {sorted(keep_residues)}")
print(f"Blocking: {len(block_residues)} -> {sorted(block_residues)}")

def block_line(l, chain, block_set):
    """Reimplementation of MEGADOCK's block script logic."""
    if not (l.startswith("ATOM") or l.startswith("HETATM")):
        return l
    if l[21] != chain:
        return l
    try:
        resnum = int(l[22:26])
    except ValueError:
        return l
    if resnum not in block_set:
        return l
    return l[0:16] + " BLK" + l[20:]

if APPLY_BLOCK:
    with open(RECEPTOR_PDB) as fin, open(BLOCKED_RECEPTOR, "w") as fout:
        for l in fin:
            fout.write(block_line(l.rstrip("\n"), chain, block_residues) + "\n")
    nblk = sum(1 for l in open(BLOCKED_RECEPTOR) if l.startswith("ATOM") and l[17:20] == "BLK")
    print(f"\nWrote {BLOCKED_RECEPTOR} ({nblk} atoms renamed BLK)")
else:
    print("\nAPPLY_BLOCK = False -> using the unblocked helix as receptor (control run)")


## 4. Set docking parameters

`-t 3 -N 10800` matches MEGADOCK's own docs for `ppiscore`-compatible runs. Default rotational sampling
(`-r 3600`, 15 deg steps). **These are identical to the full-length run so the E-scores are comparable.**

In [ ]:
# @title MEGADOCK parameters
T = 3        # predictions kept per rotation (megadock -t)
N = 10800    # total output predictions retained (megadock -N)

LIGAND_NAME = "kappa_casein"
LIGAND_PDB = "kappa_casein_frag_compact_rank004.pdb"

print(f"Receptor: {BLOCKED_RECEPTOR}")
print(f"Ligand: {LIGAND_NAME} ({LIGAND_PDB})")
print(f"t={T}, N={N}")


## 5. Run MEGADOCK

The helix is much smaller than the full-length model, so expect this to be faster. **Check the grid size
MEGADOCK reports** - it should be the same for all five ligands, otherwise the E-scores are not comparable
(this is what went wrong with kappa-casein in the full-length run).

In [ ]:
# @title Run MEGADOCK: zein helix 84-115 vs. kappa-casein fragment (64 aa, compact rank_004)
import subprocess, time

outfile = f"dock_{LIGAND_NAME}.out"
print(f"\n=== Docking zein helix 84-115 vs {LIGAND_NAME} ===")
t0 = time.time()
cmd = ["./megadock-gpu", "-R", BLOCKED_RECEPTOR, "-L", LIGAND_PDB,
       "-t", str(T), "-N", str(N), "-o", outfile]
print(" ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print("STDERR:", result.stderr)
print(f"Elapsed (wall clock, incl. overhead): {time.time()-t0:.1f} sec")

# Grid size is written on line 1 of the .out file - record it, comparability depends on it
with open(outfile) as f:
    grid_line = f.readline().strip()
print(f"\n>>> GRID / SPACING (line 1 of {outfile}): {grid_line}")
print(">>> This must match across all five ligands for the E-scores to be comparable.")


## 6. PPI score

The E-score is a Z-score: how many standard deviations the best pose sits above the mean of all 10800
retained poses. **Higher = the top pose stands out more from the random background.** It is not a binding
affinity.

In [ ]:
# @title PPI score
!./ppiscore {outfile} {N}


## 7. Generate top decoys, check restraint satisfaction, and locate PATCH-N vs PATCH-C

Same as the full-length notebook, plus one addition: for each pose we record whether the interface lands
on **PATCH-N** or **PATCH-C**.

- **PATCH-N** = `88, 92, 95` - HIS/GLN/ARG, the only charged cluster on the solvent-facing stripe
- **PATCH-C** = `106, 110, 113, 114` - ASN/TYR/GLN/GLN, the glutamine-rich cluster

Both sit on the same helical face but at opposite ends of it, about 25 A apart. Which one the ligands
prefer is a mechanistic result: PATCH-N winning implies electrostatically driven binding (testable against
the pH / ionic-strength work in 3A), PATCH-C winning implies polar/H-bonding driven binding.

In [ ]:
# @title Top-10 decoys + restraint contact check + PATCH-N / PATCH-C assignment
from Bio.PDB import PDBParser, NeighborSearch
import subprocess, csv

TOP_N = 10
CUTOFF = 5.0
LIGAND_CHAIN_LABEL = "B"

PATCH_N = {88, 92, 95}
PATCH_C = {106, 110, 113, 114}

def rechain(line, new_chain):
    if line.startswith("ATOM") or line.startswith("HETATM"):
        return line[:21] + new_chain + line[22:]
    return line

parser = PDBParser(QUIET=True)
pair_results = []

print(f"\n=== {LIGAND_NAME}: generating top {TOP_N} decoys ===")
for rank in range(1, TOP_N + 1):
    lig_decoy = f"{LIGAND_NAME}_lig.{rank}.pdb"
    complex_pdb = f"{LIGAND_NAME}_complex.{rank}.pdb"

    subprocess.run(["./decoygen", lig_decoy, LIGAND_PDB, outfile, str(rank)], check=True)

    with open(complex_pdb, "w") as out:
        with open(BLOCKED_RECEPTOR) as fin:
            for line in fin:
                if not line.startswith("END"):
                    out.write(line)
        with open(lig_decoy) as fin:
            for line in fin:
                if not line.startswith("END"):
                    out.write(rechain(line, LIGAND_CHAIN_LABEL))

    structure = parser.get_structure(complex_pdb, complex_pdb)
    atoms = list(structure.get_atoms())
    ns = NeighborSearch(atoms)

    receptor_res_in_contact = set()
    for atom in atoms:
        if atom.get_parent().get_parent().id != chain:
            continue
        for other in ns.search(atom.coord, CUTOFF):
            if other.get_parent().get_parent().id != chain:
                receptor_res_in_contact.add(atom.get_parent().id[1])
                break

    satisfied = receptor_res_in_contact & keep_residues
    fraction = len(satisfied) / len(keep_residues)
    nN = len(receptor_res_in_contact & PATCH_N)
    nC = len(receptor_res_in_contact & PATCH_C)
    patch = "N" if nN > nC else ("C" if nC > nN else ("both/tie" if nN else "neither"))

    pair_results.append((rank, len(satisfied), len(keep_residues), fraction,
                         len(receptor_res_in_contact), nN, nC, patch, complex_pdb))
    print(f"  rank {rank:2d}: {len(satisfied)}/{len(keep_residues)} kept residues in contact "
          f"({fraction:.1%}) | interface={len(receptor_res_in_contact)} res | "
          f"PATCH-N={nN}/3 PATCH-C={nC}/4 -> {patch}  ({complex_pdb})")

results_sorted = sorted(pair_results, key=lambda x: -x[3])
best = results_sorted[0]
print(f"\nBest pose by restraint satisfaction: rank {best[0]} ({best[8]}), {best[3]:.1%}, patch {best[7]}")

votes = [r[7] for r in pair_results]
print(f"Patch preference across top 10: N={votes.count('N')}  C={votes.count('C')}  "
      f"tie={votes.count('both/tie')}  neither={votes.count('neither')}")

iface_sizes = [r[4] for r in pair_results]
print(f"Interface size across top 10: min={min(iface_sizes)} max={max(iface_sizes)} "
      f"mean={sum(iface_sizes)/len(iface_sizes):.1f} residues")
print("  (full-length run gave only 3-6 - if this is similar, the receptor is still too small a target)")

summary_csv = f"{LIGAND_NAME}_restraint_summary.csv"
with open(summary_csv, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["megadock_rank", "restraints_satisfied", "restraints_total", "fraction_satisfied",
                "interface_residues", "patchN_contacts", "patchC_contacts", "patch_call", "complex_pdb"])
    for row in pair_results:
        w.writerow([row[0], row[1], row[2], f"{row[3]:.4f}", row[4], row[5], row[6], row[7], row[8]])
print(f"\nWrote {summary_csv}")


## 8. Visualize the best pose

In [ ]:
# @title Enable widgets + view the best pose (by restraint satisfaction)
from google.colab import output
output.enable_custom_widget_manager()
import nglview as nv

best_complex = results_sorted[0][8]
view = nv.show_structure_file(best_complex)
view


## 9. Download results

Results are named `megadock_zein84_115_*` so they never collide with the full-length outputs.

In [ ]:
# @title Package results for download
import os, shutil
from google.colab import files

os.makedirs("results_export", exist_ok=True)
shutil.copy(outfile, "results_export/")
shutil.copy(summary_csv, "results_export/")
shutil.copy(BLOCKED_RECEPTOR, "results_export/")   # archive the actual receptor used
for row in pair_results:
    shutil.copy(row[8], "results_export/")

zip_name = f"megadock_zein84_115_{LIGAND_NAME}_results"
shutil.make_archive(zip_name, "zip", "results_export")
files.download(f"{zip_name}.zip")

drive_results_dir = os.path.join(DRIVE_FOLDER, "results")
os.makedirs(drive_results_dir, exist_ok=True)
shutil.copy(f"{zip_name}.zip", drive_results_dir)
print(f"Also copied results to {drive_results_dir}/{zip_name}.zip")


## Interpreting the results

**Compare against the full-length run** (`../full_zein_length/`), where the five ligands gave
E-scores 5.16-5.82 with no discrimination, and interfaces of only 3-6 residues.

Three things to look for:

1. **Grid size** - must be identical across all five ligands. If one differs, its E-score is not comparable.
2. **Interface size** - if still only 3-6 residues, the 32-residue helix is too small a target and the
   result is again dominated by geometry rather than chemistry. If it rises toward 10-20, the contiguous
   stripe is behaving like a real surface.
3. **PATCH-N vs PATCH-C** - a consistent preference across ligands is a mechanistic finding. A split or
   random assignment means the docking cannot resolve sub-patches at this scale.

**Reminders**

- The E-score is a Z-score against that run's own background, **not** a binding free energy and not
  comparable to HDOCK or HADDOCK scores.
- MEGADOCK is rigid-body only. The casein fragments are intrinsically disordered and are being docked as
  single frozen conformers - a known limitation, unchanged from the full-length run.
- The helix is a proxy for one repeat unit of a crowded, curved, multi-chain nanoparticle surface.
  Avidity and multivalency are not captured; that belongs to the coarse-grained MD step (Section 3C).
- Eisenberg hydrophobic moment of this helix is only **0.067 per residue**, well below the ~0.35 threshold
  for a strongly amphipathic helix. The "this face points outward" assignment is a working hypothesis, not
  a structural fact - which is exactly what the `APPLY_BLOCK = False` control tests.

## References

- Ohue M, et al. **MEGADOCK 4.0**. *Bioinformatics*, 30(22): 3281-3283, 2014.
  https://doi.org/10.1093/bioinformatics/btu532
- MEGADOCK GitHub: https://github.com/akiyamalab/MEGADOCK
- License: CC BY-NC 4.0 - non-commercial use only without authorization from Tokyo Institute of Technology.
